<a href="https://colab.research.google.com/github/apmontesp/Landslides_-Applied-ML-Course/blob/main/notebooks/03_baseline_rf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 03: Baseline - Random Forest con HOG y Slope
 Maestría en Ingeniería - Universidad EAFIT  
**Proyecto:** Detección de deslizamientos mediante IA

Este notebook establece el rendimiento base (baseline) utilizando descriptores de textura HOG y datos de pendiente, optimizado para leer directamente desde Google Drive.

## 1. Configuración y Carga de Datos desde Drive

In [ ]:
from google.colab import drive
import os, h5py
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm.auto import tqdm
from skimage.feature import hog
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score

# Montar Google Drive
drive.mount('/content/drive')

# Configurar rutas automáticas
base_path = Path('/content/drive/MyDrive/Landslide4Sense')
detectar_img = list(base_path.glob('**/TrainData/img'))

if detectar_img:
    train_img_dir = detectar_img[0]
    train_mask_dir = detectar_img[0].parent / "mask"
    img_list = sorted(list(train_img_dir.glob("*.h5")))
    mask_list = sorted(list(train_mask_dir.glob("*.h5")))
    print(f"✅ Éxito: {len(img_list)} muestras encontradas en Drive.")
else:
    print("❌ ERROR: No se encontró la carpeta 'Landslide4Sense' en Drive.")

## 2. Extracción de Características (HOG + Slope)
Capturamos la rugosidad del terreno (HOG) y la inclinación (Banda 13).

In [ ]:
N_SAMPLES = 1000  # Ajustable según RAM
X, y = [], []

for i in tqdm(range(min(N_SAMPLES, len(img_list))), desc="Procesando parches"):
    with h5py.File(img_list[i], "r") as hf:
        patch = hf[list(hf.keys())[0]][()]
    with h5py.File(mask_list[i], "r") as hf:
        mask = hf[list(hf.keys())[0]][()]
    
    # --- CORRECCIÓN NUMPY 2.0 ---
    rgb = patch[:,:,[3,2,1]] # Bandas 4,3,2
    r_min = np.min(rgb)
    r_ptp = np.ptp(rgb)
    
    # Normalización para HOG
    rgb_norm = ((rgb - r_min) / (r_ptp + 1e-8) * 255).astype("uint8")
    
    # 1. Textura HOG
    feats_hog = hog(rgb_norm, pixels_per_cell=(16,16), cells_per_block=(2,2), 
                channel_axis=-1, feature_vector=True)
    
    # 2. Pendiente (Banda 13 -> Índice 12)
    avg_slope = np.mean(patch[:,:,12])
    
    # Vector final
    X.append(np.hstack([feats_hog, avg_slope]))
    y.append(int(np.max(mask) > 0))

X, y = np.array(X), np.array(y)
print(f"\n✓ Dataset preparado: {X.shape[0]} parches con {X.shape[1]} características.")

## 3. Entrenamiento y Validación Cruzada (Baseline)

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
f1_scores = []

print("Iniciando Entrenamiento...")
for fold, (t_idx, v_idx) in enumerate(skf.split(X, y)):
    rf = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42)
    rf.fit(X[t_idx], y[t_idx])
    
    preds = rf.predict(X[v_idx])
    f1 = f1_score(y[v_idx], preds)
    f1_scores.append(f1)
    print(f"Fold {fold+1}: F1-Score = {f1:.4f}")

print(f"\n🚀 BASELINE FINAL: {np.mean(f1_scores):.4f} ± {np.std(f1_scores):.4f}")